# Food Price Watch 02: data processing
Builds the 8-item daily panel and the driver features from `MyDrive/FoodPriceWatch_v2/raw/`, checks them, and compares them with the earlier run's panel and features.
Writes `processed/panel_8.csv`, `processed/features_8.csv`, `processed/checks.json` and `reports/tables/D1_panel_summary.csv` (rates only, no prices).
Needs notebook 01 to have run. Run top to bottom and paste the last cell's output back.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Code comes fresh from GitHub main each session; Drive holds data and outputs only.
import os
if not os.path.isdir('/content/foodprice_watch/.git'):
    !git clone --single-branch --branch main https://github.com/mrchowdhury587/foodprice_watch.git /content/foodprice_watch
else:
    !git -C /content/foodprice_watch pull --ff-only origin main
%pip install -q -r /content/foodprice_watch/requirements-colab.txt

In [ ]:
import sys, os, json
os.chdir('/content/foodprice_watch')
sys.path.insert(0, '/content/foodprice_watch/src')
CODE_COMMIT = !git rev-parse --short HEAD
CODE_COMMIT = CODE_COMMIT[0]
from foodwatch.paths import SETTINGS, DATA_ROOT, RAW, data, raw, sha256, write_csv
for folder in ['raw', 'processed', 'reports/tables']:
    (DATA_ROOT / folder).mkdir(parents=True, exist_ok=True)
for tmp in DATA_ROOT.rglob('*.tmp'):   # half-written files from an interrupted session
    tmp.unlink()
print('code commit:', CODE_COMMIT, '| data root:', DATA_ROOT)

In [ ]:
import pandas as pd
from foodwatch import load, panel, features, checks
items = SETTINGS['items']; ids = [i['id'] for i in items]; d = SETTINGS['drivers']
P = panel.build(load.prices(), ids, SETTINGS['gap_2022'], {i['id']: i['group'] for i in items})
F, avail = features.build(P, load.ramadan_eid(), load.weather(), load.fx(), weather_cols=d['weather_cols'],
                          weather_items=[i['id'] for i in items if i['weather']], fx_items=[i['id'] for i in items if i['fx']],
                          lag_days=d['weather_lag_days'], cap_days=d['carry_cap_days'])
pc = checks.panel_checks(P, len(ids))
fc = checks.feature_checks(F, avail, d['weather_lag_days'], d['carry_cap_days'])
assert pc['ok'], pc
assert fc['ok'], fc

In [ ]:
# Regression check: the same 8 items in the earlier run's panel and features (old Drive folder). Skipped if those files are not there.
from pathlib import Path
OLD = Path(SETTINGS['drive']['old_processed'])
if (OLD / 'tcb_v2_rebuilt.parquet').exists() and (OLD / 'features_v2.parquet').exists():
    rc = checks.compare_with_old(P, F, pd.read_parquet(OLD / 'tcb_v2_rebuilt.parquet'), pd.read_parquet(OLD / 'features_v2.parquet'))
    assert rc['ok'], rc
else:
    rc = {'skipped': 'old panel or features not found in ' + str(OLD)}

In [ ]:
panel_cols = ['Date', 'item_id', 'group', 'Category', 'Product', 'Unit', 'Price_Min', 'Price_Max', 'Price_Avg', 'log_price',
              'report_idx', 'days_since_prev_report', 'gap_2022', 'is_reported', 'reports_since_change']
h_panel = write_csv(P[panel_cols], data('panel'))
h_features = write_csv(F, data('features'))
summary = checks.item_summary(P)
write_csv(summary, DATA_ROOT / SETTINGS['files']['tables'] / 'D1_panel_summary.csv')
record = dict(code_commit=CODE_COMMIT, panel=dict(file=SETTINGS['files']['panel'], sha256=h_panel, **pc),
              features=dict(file=SETTINGS['files']['features'], sha256=h_features, **fc), regression_vs_old=rc,
              raw_sha256={k: sha256(raw(k)) for k in SETTINGS['raw_files']})
tmp = data('checks').with_suffix('.json.tmp'); tmp.write_text(json.dumps(record, indent=2, default=str)); tmp.replace(data('checks'))

In [ ]:
print('=== 02 data processing summary ===')
print('code commit:', CODE_COMMIT)
print('panel   :', SETTINGS['files']['panel'], '| rows', pc['rows'], '| sha256', h_panel[:16], '| checks', 'PASS' if pc['ok'] else 'FAIL')
print('features:', SETTINGS['files']['features'], '| rows', fc['rows'], '| sha256', h_features[:16], '| checks', 'PASS' if fc['ok'] else 'FAIL')
print({k: v for k, v in pc.items() if k not in ('rows', 'ok')})
print({k: v for k, v in fc.items() if k not in ('rows', 'ok', 'columns')})
print('regression vs earlier run:', 'PASS' if rc.get('ok') else rc)
if rc.get('ok'):
    print({k: v for k, v in rc.items() if k.startswith('fx_')})
print(summary.to_string(index=False))